In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:

# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
display(df.shape)
df_clean = df.dropna(subset=['Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']).copy()


df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])


display(df_clean.shape)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Task 4: Write your code here:

categorical_cols = list(df.select_dtypes(include=["object"]).columns)

for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
from pandas.io.formats.style_render import Subset
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop('Delivery_Time')  # DON'T SCALE THE TARGET
num_col = features.drop(list(df.select_dtypes(include=["object"]).columns))
scaler = StandardScaler()
df_clean[num_col] = scaler.fit_transform(df_clean[num_col])
df_clean.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:


plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=50 ,edgecolor='black')
plt.show()

In [ ]:
# Task Bonus: Write your code here: